## Workshop Lab 6: Advanced API and Batch Processing

In Notebook 5, you pulled 200 black bears for New Mexico. Notebook 6 will show you how to write a simple Python loop to batch fetch 3 different species (e.g., Prickly Pear, Ponderosa Pine, and Yucca) and how to pull data across multiple states with a single script, then export each to its own feature class automatically.

### Step 1: Setting up the parameters
We can define a list of target species and target states and then use `for` loops to collect the data.

In [ ]:
import requests
import pandas as pd
import arcpy
import os

# Define target species (using iNaturalist friendly names / scientific names)
species_targets = [
    {"common": "Ponderosa Pine", "scientific": "Pinus ponderosa"},
    {"common": "Prickly Pear", "scientific": "Opuntia"},
    {"common": "Yucca", "scientific": "Yucca"}
]

# Define state place IDs from iNaturalist (e.g., New Mexico = 9, Colorado = 34)
state_targets = [
    {"state": "New Mexico", "place_id": 9},
    {"state": "Colorado", "place_id": 34}
]

url = "https://api.inaturalist.org/v1/observations"
master_data = []

print("--- STARTING MULTI-SPECIES & MULTI-STATE BATCH HARVEST ---")

for state_info in state_targets:
    for sp in species_targets:
        print(f"Fetching {sp['common']} in {state_info['state']}...")
        
        params = {
            "q": sp["common"],
            "place_id": state_info["place_id"],
            "per_page": 50,  # Keep per-page manageable for workshop speed
            "quality_grade": "research"
        }
        
        response = requests.get(url, params=params)
        if response.status_code == 200:
            results = response.json().get('results', [])
            for obs in results:
                location = obs.get('location')
                if location:
                    lat, lon = location.split(',')
                    master_data.append({
                        'id': obs.get('id'),
                        'observed_on': obs.get('observed_on'),
                        'common_name': sp['common'],
                        'scientific_name': sp['scientific'],
                        'state': state_info['state'],
                        'latitude': float(lat),
                        'longitude': float(lon)
                    })
        else:
            print(f"  Error fetching {sp['common']} for {state_info['state']}. Status: {response.status_code}")

df_batch = pd.DataFrame(master_data)
print(f"\nSuccessfully harvested a total of {len(df_batch)} observations across all batches!")
df_batch.head()

Note: We only have 300 observations total because the code above limits data collecting to 50 per species/state combination. 

### Step 2: Cleaning and Preparing Dataframes
Now we will clean up the dates and make sure our column schema are standardized before we export.

In [ ]:
# Clean dates and extract observation year
df_batch['date_dt'] = pd.to_datetime(df_batch['observed_on'], errors='coerce')
df_batch['obs_year'] = df_batch['date_dt'].dt.year

# Check summary breakdown by state and species
print("Observations breakdown:")
print(df_batch.groupby(['state', 'common_name']).size())

### Step 3: Export to CSV and Feature Classes
Instead of exporting this as a mixed csv or feature class, let's use a loop to split the dataframe by species and then export each into the the ArcPro geodatabase!

In [ ]:
# Setup working directories
working_dir = os.path.abspath(os.path.join(os.getcwd(), "..", "04_data", "working"))
os.makedirs(working_dir, exist_ok=True)

gdb_path = os.path.abspath(os.path.join(os.getcwd(), "..", "03_pro_project", "IntroToPythonAndArcade.gdb"))
arcpy.env.workspace = gdb_path
arcpy.env.overwriteOutput = True

print(f"\n--- EXPORTING SEPARATE FEATURE CLASSES TO GDB ---")
print(f"Target Workspace: {gdb_path}\n")

# Loop through each unique species and create individual outputs
for species_name in df_batch['common_name'].unique():
    # Filter dataframe for this specific species
    df_sub = df_batch[df_batch['common_name'] == species_name].copy()
    
    # Create a safe file/table name (remove spaces)
    safe_name = species_name.replace(" ", "_").lower()
    csv_filename = f"batch_{safe_name}.csv"
    csv_path = os.path.join(working_dir, csv_filename)
    
    # Save CSV
    df_sub.to_csv(csv_path, index=False)
    
    # Convert CSV to Feature Class in Pro GDB
    out_fc = f"fc_{safe_name}"
    arcpy.management.XYTableToPoint(
        in_table=csv_path,
        out_feature_class=out_fc,
        x_field="longitude",
        y_field="latitude",
        coordinate_system=arcpy.SpatialReference(4326)
    )
    
    print(f"Created feature class '{out_fc}' with {len(df_sub)} records.")

print("\nBatch processing complete! Check your ArcGIS Pro Catalog pane to see your new feature classes.")

You should see three new point feature classes in your geodatabase!

## Next Step: Open 07_InteractiveMapDashboards.ipynb